# Ejercicio: Implementar un método callbacks usando el conjuntod e datos MNIST

Hemos aprendido a clasificar usando Fashion MNIST, un conjunto de datos que contiene prendas de vestir. Hay otro conjunto de datos similar llamado MNIST que tiene elementos de escritura a mano: los dígitos del 0 al 9.

Escriba un clasificador MNIST con una precisión del 99 % o más, y lo haga sin un número fijo de épocas; es decir, debe dejar de entrenar una vez que alcance ese nivel de precisión.

Algunas notas:
1. Dada la arquitectura de la red, deberemos tener éxito en menos de 10 épocas.
2. Cuando alcance el 99 % o más, debes imprimir la cadena "Alcanzó el 99 % de precisión, por lo que entrenamiento cancelado". y dejar de entrenar.

In [5]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

In [6]:
from tensorflow.keras.datasets import mnist

Comenzamos cargando los datos pero debemos tener un par de cosas a tener en cuenta:

- El archivo `mnist.npz` lo debemos descargar y subir a nuestra unidad de Google Drive, a partir de aquí debemos indicar la ruta en la que se encuentra en el método ya implementado en la API de Keras.

- `load_data` devuelve los conjuntos de entrenamiento y test en forma de tuplas `(x_train, y_train), (x_test, y_test)` pero en este ejercicio solo necesitaramos el conjunto de entrenamiento por lo que podemos ignorar la segunda tupla.

Echa un vistazo al conjunto de datos

In [7]:
# Cargamos el conjunto de datos MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Ahora es el momento de crear nuestro propia clase de callbacks.

In [ ]:
class myNewCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Comprobamos el valor de la loss
        accuracy_actual = logs.get('accuracy')

        if accuracy_actual is not None and accuracy_actual >= 0.99:
            print(f"\n Se detiene el entrenamiento: accuaracy = {accuracy_actual:.4f} (mayor o igual  al 99%)")
            self.model.stop_training = True

Ahora que has definido tu clase callbacks desarrolla `entrenar_mnist` para entrenar el modelo

In [ ]:
def entrenar_mnist(x_train, y_labels):

    # Normalización
    x_train = x_train.astype("float32") / 255.0

    # Aplanado (28x28 → 784)
    x_train = x_train.reshape((x_train.shape[0], 28 * 28))

    # One-hot encoding
    y_labels = tf.keras.utils.to_categorical(y_labels, 10)

    # Definición del modelo
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(10, activation='softmax')
    ])

    # Compilación
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Instanciar correctamente el callback
    callback_instance = myNewCallback()

    # ENTRENAMIENTO con tu callback
    model.fit(
        x_train,
        y_labels,
        epochs=10,
        batch_size=32,
        callbacks=[callback_instance]
    )

    return model


Llama al método pasándole el conjunto de datos

In [ ]:
modelo = entrenar_mnist(X_train, y_train)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8779 - loss: 0.4326
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9647 - loss: 0.1236
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9761 - loss: 0.0804
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9822 - loss: 0.0567
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9873 - loss: 0.0430
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9896 - loss: 0.0336
Epoch 7/10
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9919 - loss: 0.0270
 Se detiene el entrenamiento: accuaracy = 0.9906 (mayor o igual  al 99%)
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9919 - loss: 0.0270


Si ve el mensaje "Alcanzó el 99% de precisiónn por lo que entrenamiento cancelado" impreso después de menos de 10 épocas es una señal de buen trabajo.